In [5]:
%pip install streamlit mlxtend pandas

In [6]:
%%writefile app.py

import streamlit as st
import pandas as pd

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules


# --------------------------------------------------
# PAGE CONFIGURATION
# --------------------------------------------------

st.set_page_config(
    page_title="Convenience Store Market Basket Pattern Explorer",
    layout="wide"
)

st.title("Convenience Store Market Basket Pattern Explorer")

st.write(
    """
    Upload a convenience-store transaction dataset to discover
    frequently purchased product combinations and association rules
    using the Apriori algorithm.
    """
)


# --------------------------------------------------
# SIDEBAR SETTINGS
# --------------------------------------------------

st.sidebar.header("Analysis Settings")

min_support = st.sidebar.slider(
    "Minimum Support",
    min_value=0.05,
    max_value=0.50,
    value=0.10,
    step=0.05
)

min_confidence = st.sidebar.slider(
    "Minimum Confidence",
    min_value=0.10,
    max_value=1.00,
    value=0.80,
    step=0.05
)

st.sidebar.write(
    f"Minimum Support: {min_support:.2f}"
)

st.sidebar.write(
    f"Minimum Confidence: {min_confidence:.2f}"
)


# --------------------------------------------------
# FILE UPLOAD
# --------------------------------------------------

uploaded_file = st.file_uploader(
    "Upload Transaction CSV",
    type=["csv"]
)


# --------------------------------------------------
# RUN ANALYSIS
# --------------------------------------------------

if uploaded_file is not None:

    try:

        # ------------------------------------------
        # LOAD DATASET
        # ------------------------------------------

        df = pd.read_csv(uploaded_file)

        required_columns = [
            "Transaction_ID",
            "Purchased_Items"
        ]


        # ------------------------------------------
        # VALIDATE COLUMNS
        # ------------------------------------------

        if not all(
            column in df.columns
            for column in required_columns
        ):
            st.error(
                "The CSV must contain the columns "
                "'Transaction_ID' and 'Purchased_Items'."
            )
            st.stop()


        # ------------------------------------------
        # VALIDATE EMPTY DATA
        # ------------------------------------------

        if df.empty:
            st.error(
                "The uploaded CSV contains no transactions."
            )
            st.stop()


        if df[required_columns].isnull().any().any():
            st.error(
                "The dataset contains missing Transaction IDs "
                "or Purchased Items."
            )
            st.stop()


        # ------------------------------------------
        # DATASET INFORMATION
        # ------------------------------------------

        st.success("Dataset loaded successfully.")

        st.subheader("Dataset Information")

        col1, col2 = st.columns(2)

        with col1:
            st.metric(
                "Transactions",
                len(df)
            )

        with col2:
            st.metric(
                "Dataset Columns",
                len(df.columns)
            )


        st.subheader("Dataset Preview")

        st.dataframe(
            df,
            use_container_width=True
        )


        # ------------------------------------------
        # PREPARE TRANSACTIONS
        # ------------------------------------------

        transactions = (
            df["Purchased_Items"]
            .str.split(",")
            .apply(
                lambda items: [
                    item.strip()
                    for item in items
                    if item.strip()
                ]
            )
            .tolist()
        )


        # ------------------------------------------
        # TRANSACTION ENCODING
        # ------------------------------------------

        encoder = TransactionEncoder()

        encoded_array = (
            encoder
            .fit(transactions)
            .transform(transactions)
        )

        basket_df = pd.DataFrame(
            encoded_array,
            columns=encoder.columns_
        )


        # ------------------------------------------
        # APRIORI
        # ------------------------------------------

        frequent_itemsets = apriori(
            basket_df,
            min_support=min_support,
            use_colnames=True,
            max_len=3
        )


        # ------------------------------------------
        # CHECK ITEMSET RESULTS
        # ------------------------------------------

        if frequent_itemsets.empty:

            st.warning(
                "No frequent itemsets were found using the "
                "selected minimum support. Try lowering "
                "the support threshold."
            )

            st.stop()


        # ------------------------------------------
        # SUPPORT COUNT
        # ------------------------------------------

        frequent_itemsets["support_count"] = (
            frequent_itemsets["support"]
            * len(df)
        ).round().astype(int)


        # ------------------------------------------
        # FORMAT ITEMSETS
        # ------------------------------------------

        def format_itemset(itemset):
            return ", ".join(
                sorted(itemset)
            )


        itemset_results = frequent_itemsets.copy()

        itemset_results["itemsets"] = (
            itemset_results["itemsets"]
            .apply(format_itemset)
        )

        itemset_results = itemset_results[
            [
                "itemsets",
                "support",
                "support_count"
            ]
        ]

        itemset_results = (
            itemset_results
            .sort_values(
                by="support_count",
                ascending=False
            )
            .reset_index(drop=True)
        )


        # ------------------------------------------
        # DISPLAY FREQUENT ITEMSETS
        # ------------------------------------------

        st.subheader("Frequent Itemsets")

        st.write(
            """
            Frequent itemsets show products or product
            combinations that appear often enough to meet
            the selected minimum support.
            """
        )

        st.dataframe(
            itemset_results,
            use_container_width=True
        )


        # ------------------------------------------
        # ASSOCIATION RULES
        # ------------------------------------------

        rules = association_rules(
            frequent_itemsets,
            metric="confidence",
            min_threshold=min_confidence
        )


        if rules.empty:

            st.warning(
                "Frequent itemsets were found, but no "
                "association rules meet the selected "
                "minimum confidence. Try lowering the "
                "confidence threshold."
            )

        else:

            rule_results = rules[
                [
                    "antecedents",
                    "consequents",
                    "support",
                    "confidence",
                    "lift"
                ]
            ].copy()

            rule_results["support_count"] = (
                rule_results["support"] * len(df)
            ).round().astype(int)

            rule_results["antecedents"] = (
                rule_results["antecedents"]
                .apply(format_itemset)
            )

            rule_results["consequents"] = (
                rule_results["consequents"]
                .apply(format_itemset)
            )

            rule_results = (
                rule_results
                .sort_values(
                    by="lift",
                    ascending=False
                )
                .reset_index(drop=True)
            )

            st.subheader("Association Rules")

            st.write(
                """
                Association rules show relationships between
                products using support, confidence, and lift.
                """
            )

            st.dataframe(
                rule_results,
                use_container_width=True
            )

            st.subheader("How to Read the Results")

            st.write(
                """
                *Support* — how frequently the product
                combination appears in all transactions.

                *Confidence* — how often the consequent
                appears when the antecedent is purchased.

                *Lift* — how strongly the products are
                associated compared with random occurrence.

                *Support Count* — the actual number of
                transactions containing the combination.
                """
            )

            rules_csv = (
                rule_results
                .to_csv(index=False)
                .encode("utf-8")
            )

            st.download_button(
                label="Download Association Rules",
                data=rules_csv,
                file_name="association_rules_results.csv",
                mime="text/csv"
            )

        itemsets_csv = (
            itemset_results
            .to_csv(index=False)
            .encode("utf-8")
        )

        st.download_button(
            label="Download Frequent Itemsets",
            data=itemsets_csv,
            file_name="frequent_itemsets_results.csv",
            mime="text/csv"
        )

    except Exception as error:

        st.error(
            f"Unable to process the dataset: {error}"
        )

else:

    st.info(
        "Upload CS_Transactions.csv to begin "
        "the market basket analysis."
    )


Overwriting app.py


In [7]:
!ls -lh app.py
!wc -l app.py


-rw-r--r-- 1 root root 9.7K Oct 15 01:14 app.py
407 app.py


In [8]:
!python -m py_compile app.py


In [9]:
!streamlit run app.py --server.port 8501 > /content/streamlit.log 2>&1 &

In [10]:
!cat /content/streamlit.log

In [11]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501, height=800)

<IPython.core.display.Javascript object>

In [12]:
from google.colab import output

output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [13]:
!streamlit run app.py --server.port 8501 > /content/streamlit.log 2>&1 &

In [14]:
!cat /content/streamlit.log

In [15]:
!curl -I http://localhost:8501

curl: (7) Failed to connect to localhost port 8501 after 1 ms: Couldn't connect to server


In [16]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [17]:
!./cloudflared --version

cloudflared version 2026.10.0 (built 2026-10-05-17:37 UTC)


In [ ]:
!./cloudflared tunnel --url http://localhost:8501

2026-10-15T01:14:48Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-15T01:14:48Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-15T01:14:51Z INF +--------------------------------------------------------------------------------------------+
2026-10-15T01:14:51Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-15T01:14:51Z INF |  https://ratio-customs-thousands-colleagues.trycloudfl